# Harmonization: UNSW-NB15 → ตารางกลาง

ใน notebook นี้แสดงการ map ข้อมูลไว้เป็นขั้น ๆ โดยตรง ไม่เรียก `harmonize_dataset()` หรือฟังก์ชัน schema จากไฟล์ภายนอก

เราอ่านผล cleansing ของ official Train/Test แล้วแยกเป็น `features`, `labels`, `metadata` ซึ่งใช้ `record_id` เชื่อมกัน คอลัมน์ที่หาไม่พบจะเป็น missing ไม่เดาจากตำแหน่ง ส่วน label ที่หายหรือไม่สอดคล้องจะทำให้หยุด

> แนะนำให้รัน EDA และ Cleansing ก่อน ตัวอย่าง output เป็น CSV และใช้เป็น input ของ `data-pipeline/build_quad_dataset.py`

## 1. ตรวจผล cleansing, raw manifest และรุ่น contract

ยืนยันว่า raw ยังตรงกับ checksum ที่บันทึกตอน cleansing และได้ผล CSV ของ Train/Test ครบก่อนเริ่มแปลง เพื่อป้องกันการใช้ผลเก่ากับข้อมูลต้นฉบับคนละรุ่น

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
project_root = next((path for path in (current, *current.parents)
                     if (path / 'data-pipeline' / 'sub-dataset' / 'UNSW-NB15' / 'README.md').is_file()), None)
if project_root is None:
    raise FileNotFoundError('เปิด notebook จากภายในโฟลเดอร์โปรเจกต์ CSI_ML_Final')

pipeline_dir = project_root / 'data-pipeline'
dataset_dir = pipeline_dir / 'sub-dataset' / 'UNSW-NB15'
clean_dir = dataset_dir / 'output' / 'cleansing'
output_dir = dataset_dir / 'output' / 'harmonized'
split_files = {
    'train': clean_dir / 'UNSW_NB15_training-set.csv',
    'test': clean_dir / 'UNSW_NB15_testing-set.csv',
}
missing = [path.name for path in split_files.values() if not path.is_file()]
manifest_path = clean_dir / 'source_manifest.csv'
if missing or not manifest_path.is_file():
    raise FileNotFoundError(
        f'ไม่พบผล cleansing/manifest: {missing}; รัน UNSW-NB15/cleansing.ipynb ก่อน'
    )

def sha256_file(file_path):
    digest = hashlib.sha256()
    with file_path.open('rb') as file:
        for block in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

raw_dir = dataset_dir / 'raw'
raw_files = sorted(path for path in raw_dir.rglob('*')
                   if path.is_file() and path.suffix.lower() in {'.csv', '.gz', '.txt'})
current_manifest = pd.DataFrame([
    {'file': path.relative_to(raw_dir).as_posix(), 'bytes': path.stat().st_size,
     'sha256': sha256_file(path)} for path in raw_files
], columns=['file', 'bytes', 'sha256']).sort_values('file').reset_index(drop=True)
saved_manifest = pd.read_csv(manifest_path).sort_values('file').reset_index(drop=True)
if not current_manifest['file'].equals(saved_manifest['file']) or not current_manifest['sha256'].equals(saved_manifest['sha256']):
    raise ValueError('raw เปลี่ยนหลัง cleansing; รัน cleansing ใหม่ก่อน harmonization')

registry = json.loads((pipeline_dir / 'CENTRAL' / 'dataset_registry.json').read_text(encoding='utf-8'))
contract = json.loads((pipeline_dir / 'CENTRAL' / 'feature_contract.json').read_text(encoding='utf-8'))
print('ไฟล์พร้อม:', {name: path.name for name, path in split_files.items()})
print('raw checksum ตรงกับ manifest; dataset version:', registry['datasets']['UNSW-NB15']['version'])

ไฟล์พร้อม: {'train': 'UNSW_NB15_training-set.csv', 'test': 'UNSW_NB15_testing-set.csv'}
raw checksum ตรงกับ manifest; dataset version: UNSW-NB15 official train/test CSV release


## 2. กติกาแปลงที่ใช้กับทุกแถว

ค้นคอลัมน์จากชื่อ header หลัง normalize BOM/ช่องว่าง ไม่อาศัยลำดับคอลัมน์ จากนั้นแปลงค่าตัวเลข/หมวดหมู่ และเก็บค่าที่ใช้ไม่ได้เป็น missing พร้อม quality flag

In [2]:
def find_column(frame, *candidate_names):
    normalized = {str(name).replace('\ufeff', '').strip().lower(): name
                  for name in frame.columns}
    for candidate in candidate_names:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]
    return None

def nonnegative_number(frame, column_name):
    if column_name is None or column_name not in frame.columns:
        empty_values = pd.Series(np.nan, index=frame.index, dtype='float64')
        no_invalid_values = pd.Series(False, index=frame.index, dtype=bool)
        return empty_values, no_invalid_values
    original = frame[column_name]
    number = pd.to_numeric(original, errors='coerce').astype('float64')
    invalid = original.notna() & (number.isna() | ~np.isfinite(number) | number.lt(0))
    number = number.replace([np.inf, -np.inf], np.nan)
    number.loc[number < 0] = np.nan
    return number, invalid.astype(bool)

def optional_number(frame, column_name):
    values, _ = nonnegative_number(frame, column_name)
    return values

def map_attack_category(raw_label, source_category):
    label = '' if pd.isna(raw_label) else str(raw_label).strip().lower()
    category = '' if pd.isna(source_category) else str(source_category).strip().lower()
    value = category or label
    if label in {'0', 'normal', 'benign', 'background'} or value in {'normal', 'benign', 'background'}:
        return 'Normal'
    category_map = {
        'dos': 'DoS/DDoS', 'ddos': 'DoS/DDoS',
        'reconnaissance': 'Probe/Recon', 'recon': 'Probe/Recon',
        'exploits': 'Exploit/U2R', 'exploit': 'Exploit/U2R', 'shellcode': 'Exploit/U2R',
        'backdoors': 'Botnet/Backdoor', 'backdoor': 'Botnet/Backdoor',
        'botnet': 'Botnet/Backdoor', 'worms': 'Botnet/Backdoor',
        'fuzzers': 'Generic/Fuzzer', 'fuzzer': 'Generic/Fuzzer',
        'generic': 'Generic/Fuzzer', 'analysis': 'Probe/Recon',
    }
    return category_map.get(value, 'Other attack')

print('กติกาพร้อม: ชื่อคอลัมน์ → ค่ากลาง; label → taxonomy กลาง')

กติกาพร้อม: ชื่อคอลัมน์ → ค่ากลาง; label → taxonomy กลาง


## 3. แปลงหนึ่ง split เป็น features, labels และ metadata

ตารางทั้งสามต้องมี record_id เดียวกันและเรียงเหมือนกัน `source_split` มาจากไฟล์ official ที่อ่านอยู่; `capture_time` แปลงจาก Unix seconds หากมี `stime`

In [3]:
def harmonize_split(clean_frame, split_name, source_file):
    frame = clean_frame.copy()
    label_column = find_column(frame, 'label')
    category_column = find_column(frame, 'attack_cat')
    if label_column is None:
        raise ValueError(f'{source_file}: ไม่มีคอลัมน์ label')

    raw_label = frame[label_column].astype('string').str.strip()
    if raw_label.isna().any() or raw_label.eq('').any():
        raise ValueError(f'{source_file}: พบ label ว่าง; ไม่จัด label ที่หายเป็น attack')
    category = (frame[category_column].astype('string').str.strip()
                if category_column else pd.Series(pd.NA, index=frame.index, dtype='string'))
    original_binary = pd.to_numeric(raw_label, errors='coerce')
    if original_binary.isna().any() or not original_binary.isin([0, 1]).all():
        raise ValueError(f'{source_file}: label ต้องมีเฉพาะ 0/1')
    category_is_normal = category.str.lower().isin(['normal', 'benign'])
    if (category_is_normal.to_numpy() != original_binary.eq(0).to_numpy()).any():
        raise ValueError(f'{source_file}: label 0/1 ไม่ตรงกับ attack_cat normal/attack')
    attack_category = [map_attack_category(label, name)
                       for label, name in zip(raw_label, category)]

    duration, invalid_duration = nonnegative_number(frame, find_column(frame, 'dur'))
    forward_bytes, invalid_forward = nonnegative_number(frame, find_column(frame, 'sbytes'))
    backward_bytes, invalid_backward = nonnegative_number(frame, find_column(frame, 'dbytes'))
    protocol_column = find_column(frame, 'proto')
    if protocol_column:
        protocol = frame[protocol_column].astype('string').str.strip().str.lower()
        protocol = protocol.replace({'6': 'tcp', '17': 'udp', '1': 'icmp'})
        protocol = protocol.where(protocol.isna() | protocol.isin(['tcp', 'udp', 'icmp']), 'other')
    else:
        protocol = pd.Series(pd.NA, index=frame.index, dtype='string')

    service_column = find_column(frame, 'service')
    if service_column:
        service_text = frame[service_column].astype('string').str.strip().str.lower()
        service_groups = {'http':'http', 'www':'http', 'https':'https_ssl', 'ssl':'https_ssl',
                          'ftp':'ftp', 'ftp_data':'ftp', 'ftp-data':'ftp', 'ssh':'ssh', 'smtp':'smtp',
                          'domain':'dns', 'domain_u':'dns', 'dns':'dns',
                          '-':'other', 'other':'other'}
        service = service_text.map(service_groups).fillna('other').where(service_text.notna())
    else:
        port_column = find_column(frame, 'Destination Port', 'dst port', 'dsport')
        if port_column:
            destination_port = pd.to_numeric(frame[port_column], errors='coerce')
            service = pd.Series('other', index=frame.index, dtype='string')
            port_groups = {80:'http', 443:'https_ssl', 53:'dns', 20:'ftp', 21:'ftp',
                           22:'ssh', 25:'smtp', 465:'smtp', 587:'smtp'}
            for port, group in port_groups.items():
                service.loc[destination_port.eq(port)] = group
            service = service.where(destination_port.notna(), pd.NA)
        else:
            service = pd.Series(pd.NA, index=frame.index, dtype='string')

    state_column = find_column(frame, 'state')
    if state_column:
        state_text = frame[state_column].astype('string').str.strip().str.lower()
        state_groups = {'sf':'established', 'fin':'established', 'con':'established',
                        's0':'no-reply', 'int':'no-reply', 'req':'no-reply',
                        'rej':'reset', 'rsto':'reset', 'rstr':'reset', 'rstos0':'reset', 'rst':'reset'}
        connection_state = state_text.map(state_groups).fillna('other').where(state_text.notna())
    else:
        connection_state = pd.Series(pd.NA, index=frame.index, dtype='string')

    packets_forward = optional_number(frame, find_column(frame, 'spkts'))
    packets_backward = optional_number(frame, find_column(frame, 'dpkts'))
    packet_total = packets_forward + packets_backward
    mean_packet_size = (forward_bytes + backward_bytes).div(packet_total.where(packet_total > 0))
    ttl_forward = optional_number(frame, find_column(frame, 'sttl'))
    ttl_backward = optional_number(frame, find_column(frame, 'dttl'))
    window_forward = optional_number(frame, find_column(frame, 'swin'))
    window_backward = optional_number(frame, find_column(frame, 'dwin'))

    row_ids = frame['orig_row_id'].astype('string')
    record_ids = [f'UNSW-NB15:{row_id}' for row_id in row_ids.astype(str)]
    features = pd.DataFrame({
        'record_id': record_ids, 'duration': duration.astype('float64'),
        'protocol': protocol.astype('string'), 'bytes_forward': forward_bytes.astype('float64'),
        'bytes_backward': backward_bytes.astype('float64'), 'service_group': service.astype('string'),
        'conn_state_group': connection_state.astype('string'),
        'packets_forward': packets_forward.astype('float64'),
        'packets_backward': packets_backward.astype('float64'),
        'mean_packet_size': mean_packet_size.astype('float64'),
        'ttl_forward': ttl_forward.astype('float64'), 'ttl_backward': ttl_backward.astype('float64'),
        'tcp_window_forward': window_forward.astype('float64'),
        'tcp_window_backward': window_backward.astype('float64'),
    })
    nsl_only_columns = ['nsl_hot', 'nsl_logged_in', 'nsl_failed_logins', 'nsl_root_shell',
                        'nsl_count_2s', 'nsl_srv_count_2s', 'nsl_same_srv_rate_2s', 'nsl_serror_rate_2s']
    for column in nsl_only_columns:
        features[column] = np.nan

    labels = pd.DataFrame({
        'record_id': record_ids, 'label_raw': raw_label.reset_index(drop=True),
        'label_source_category': category.reset_index(drop=True),
        'attack_category': attack_category,
        'label_binary': original_binary.astype('int8').reset_index(drop=True),
    })

    timestamp_column = find_column(frame, 'stime')
    if timestamp_column:
        timestamp_seconds = pd.to_numeric(frame[timestamp_column], errors='coerce')
        capture_time = pd.to_datetime(timestamp_seconds, unit='s', errors='coerce', utc=True)
    else:
        capture_time = pd.Series(pd.NaT, index=frame.index, dtype='datetime64[ns, UTC]')
    capture_time_text = capture_time.dt.strftime('%Y-%m-%dT%H:%M:%SZ').astype('string')
    duplicate_flag = (frame['_duplicate_flag'].fillna(False).astype(bool)
                      if '_duplicate_flag' in frame else pd.Series(False, index=frame.index))
    metadata = pd.DataFrame({
        'record_id': record_ids, 'source_dataset': 'UNSW-NB15',
        'orig_row_id': row_ids.to_numpy(), 'source_split': split_name,
        'capture_time': capture_time_text,
        'split_group': capture_time.dt.strftime('%Y-%m-%d').astype('string'),
        'dedup_flag': duplicate_flag.to_numpy(), 'sample_weight': 1.0,
        'has_duration': features['duration'].notna().to_numpy(),
        'invalid_duration': invalid_duration.to_numpy(),
        'has_protocol': features['protocol'].notna().to_numpy(),
        'has_directional_bytes': features[['bytes_forward', 'bytes_backward']].notna().all(axis=1).to_numpy(),
        'invalid_bytes_forward': invalid_forward.to_numpy(),
        'invalid_bytes_backward': invalid_backward.to_numpy(),
        'has_capture_time': capture_time_text.notna().to_numpy(),
        'has_service': features['service_group'].notna().to_numpy(),
        'has_connection_state': features['conn_state_group'].notna().to_numpy(),
        'has_packet_stats': features[['packets_forward', 'packets_backward', 'mean_packet_size']].notna().any(axis=1).to_numpy(),
        'has_tcp_details': features[['ttl_forward', 'ttl_backward', 'tcp_window_forward', 'tcp_window_backward']].notna().any(axis=1).to_numpy(),
        'has_nsl_content': False, 'has_nsl_window': False, 'is_kddtest21_view': False,
    })
    return features, labels, metadata, invalid_duration, invalid_forward, invalid_backward

## 4.1 รวมตารางของแต่ละ split

แปลงข้อมูล train และ test ตามกติกาเฉพาะชุด แล้วรวม features, labels และ metadata เพื่อใช้กฎกลาง


In [ ]:
features_parts, labels_parts, metadata_parts = [], [], []
quality_rows = []

for split_name, clean_path in split_files.items():
    clean_frame = pd.read_csv(clean_path)
    feature_table, label_table, metadata_table, bad_duration, bad_forward, bad_backward = (
        harmonize_split(clean_frame, split_name, clean_path.name)
    )
    features_parts.append(feature_table)
    labels_parts.append(label_table)
    metadata_parts.append(metadata_table)
    quality_rows.append({
        'source_dataset': 'UNSW-NB15', 'split': split_name,
        'rows': len(feature_table),
        'duplicate_rows': int(metadata_table['dedup_flag'].sum()),
        'invalid_duration': int(bad_duration.sum()),
        'invalid_bytes_forward': int(bad_forward.sum()),
        'invalid_bytes_backward': int(bad_backward.sum()),
    })

features = pd.concat(features_parts, ignore_index=True)
labels = pd.concat(labels_parts, ignore_index=True)
metadata = pd.concat(metadata_parts, ignore_index=True)

PIPELINE_DIR = pipeline_dir


## 4.2 เตรียมกฎปรับหมวดหมู่และตรวจคุณภาพ

กำหนดชื่อ protocol, service และ connection state ที่ใช้ร่วมกัน พร้อมกฎทำเครื่องหมายแถวที่มีข้อมูลขัดกัน


In [ ]:
CLASS_NAMES = [
    "Benign / Normal",
    "Denial of Service (DoS)",
    "Reconnaissance & Probing",
    "Exploits & PrivEsc",
    "Credential Abuse (R2L)",
    "Malware, C2 & Botnet",
]


def require(condition, message):
    if not condition:
        raise ValueError(message)


SERVICE_ALIASES = {
    'http':'http', 'www':'http', 'http_443':'https_ssl',
    'https':'https_ssl', 'ssl':'https_ssl',
    'ftp':'ftp', 'ftp_data':'ftp', 'ftp-data':'ftp',
    'ssh':'ssh', 'smtp':'smtp', 'domain':'dns', 'domain_u':'dns', 'dns':'dns',
}


STATE_ALIASES = {
    'sf':'established', 's1':'established', 's2':'established', 's3':'established',
    'fin':'established', 'con':'established', 'established':'established',
    's0':'no-reply', 'req':'no-reply', 'int':'no-reply', 'no-reply':'no-reply',
    'rej':'reset', 'rsto':'reset', 'rstr':'reset', 'rstos0':'reset',
    'rst':'reset', 'reset':'reset',
}


QUALITY_FLAGS = ['invalid_required_value', 'bytes_packet_conflict', 'state_counter_conflict']


def normalize_boolean_metadata(metadata):
    columns=[c for c in metadata if c.startswith(('has_','invalid_')) or c in QUALITY_FLAGS+['quality_ok','service_identified','state_reclassified','dedup_flag','is_kddtest21_view']]
    for col in columns:
        values=metadata[col].astype('string').str.lower()
        if not values.isin(['true','false','1','0']).all():raise ValueError(f'Invalid boolean metadata: {col}')
        metadata[col]=values.map({'true':True,'false':False,'1':True,'0':False}).astype(bool)
    return metadata


def text(values):
    return values.astype('string').str.strip().str.lower().replace({'':'', '-':'-'})


def normalize_protocol(values):
    values=text(values).replace({'6':'tcp','6.0':'tcp','17':'udp','17.0':'udp',
                                 '1':'icmp','1.0':'icmp','58':'icmp','58.0':'icmp','ipv6-icmp':'icmp'})
    present=values.notna() & ~values.isin(['','-'])
    return values.where(values.isin(['tcp','udp','icmp']),'other').where(present,pd.NA)


def port_to_service(value):
    if pd.isna(value) or str(value).strip()=='':return 'other'
    token=str(value).strip()
    try:
        number=int(token,16) if token.lower().startswith('0x') else float(token)
    except (ValueError,TypeError) as error:
        raise ValueError(f'Invalid destination port: {value!r}') from error
    if not np.isfinite(number) or number!=int(number) or not 0<=number<=65535:
        raise ValueError(f'Invalid destination port: {value!r}')
    port=int(number)
    return {53:'dns',20:'ftp',21:'ftp',80:'http',8080:'http',443:'https_ssl',
            25:'smtp',465:'smtp',587:'smtp',22:'ssh'}.get(port,'other')


def quality_flags(features, metadata):
    f=features
    invalid=pd.Series(False,index=f.index)
    for col in ['duration','bytes_forward','bytes_backward']:
        values=pd.to_numeric(f[col],errors='coerce')
        invalid |= values.isna() | ~np.isfinite(values) | values.lt(0)
    invalid |= f.protocol.isna() | f.protocol.astype('string').str.strip().isin(['','-'])
    packet_conflict=pd.Series(False,index=f.index)
    for direction in ['forward','backward']:
        col='packets_'+direction
        if col in f:
            values=pd.to_numeric(f[col],errors='coerce')
            packet_conflict |= values.eq(0) & f['bytes_'+direction].gt(0)
            invalid |= f[col].notna() & (~np.isfinite(values) | values.lt(0))
    raw_state=text(metadata.get('source_connection_state',pd.Series(pd.NA,index=f.index,dtype='string')))
    has_response=f.bytes_backward.gt(0)
    if 'packets_backward' in f:has_response |= f.packets_backward.gt(0)
    # REQ/INT can have response traffic. Literal no-reply/S0 with a response is
    # contradictory source evidence; do not silently erase that evidence.
    state_conflict=raw_state.isin(['no-reply','s0']) & has_response
    return pd.DataFrame({'invalid_required_value':invalid,
                         'bytes_packet_conflict':packet_conflict,
                         'state_counter_conflict':state_conflict},index=f.index)


## 4.3 กำหนดการแปลง label เป็นคลาสกลาง

จับคู่ label ต้นทางกับรหัสคลาสและตรวจว่าตรงกับ label แบบปกติหรือโจมตี


In [ ]:
TACTICS = ['Baseline Operations','TA0040: Impact','TA0043/TA0007: Recon & Discovery','TA0001/TA0004: Initial Access & PrivEsc','TA0006: Credential Access','TA0011/TA0008: C2 & Lateral Movement']


COMMON_MAPPING = {'normal':0,'benign':0,'dos':1,'dos/ddos':1,'ddos':1,'probe':2,'probe/recon':2,'reconnaissance':2,'analysis':2,'exploits':3,'exploit/u2r':3,'shellcode':3,'u2r':3,'r2l':4,'brute-force/r2l':4,'generic':5,'generic/fuzzer':5,'fuzzers':5,'backdoor':5,'backdoors':5,'botnet/backdoor':5,'worms':5}


SOURCE_MAPPING = {
    'NSL-KDD': {k:COMMON_MAPPING[k] for k in ['normal','dos','probe','r2l','u2r']},
    'UNSW-NB15': {k:COMMON_MAPPING[k] for k in ['normal','analysis','backdoor','backdoors','dos','exploits','fuzzers','generic','reconnaissance','shellcode','worms']},
    'CIC-IDS2017': {**COMMON_MAPPING, 'dos hulk':1,'dos goldeneye':1,'dos slowloris':1,'dos slowhttptest':1,'heartbleed':3,'portscan':2,'ftp-patator':4,'ssh-patator':4,'web attack - brute force':4,'web attack - xss':3,'web attack - sql injection':3,'web attack brute force':4,'web attack xss':3,'web attack sql injection':3,'infiltration':3,'bot':5},
    # Project-level category grouping, not per-record MITRE attribution.
    'BoT-IoT': {**COMMON_MAPPING,'theft':5,'data_theft':5,'bruteforce':4},
}


def map_labels(labels, metadata):
    result=labels.copy(); ids=pd.Series(index=labels.index,dtype='int64')
    for source,index in metadata.groupby('source_dataset').groups.items():
        require(source in SOURCE_MAPPING,f'Unsupported source {source}')
        values=labels.loc[index,'label_source_category'].astype('string').str.strip().str.lower().str.replace(r'\s+',' ',regex=True)
        mapped=values.map(SOURCE_MAPPING[source])
        require(mapped.notna().all(),f'Unmapped labels in {source}: {values[mapped.isna()].value_counts().to_dict()}')
        ids.loc[index]=mapped.astype(int)
    ids=ids.astype(int)
    require(labels.label_binary.eq(ids.ne(0).astype(int)).all(),'Mapped categories contradict original binary truth')
    result['attack_category_id']=ids;result['attack_category']=ids.map(dict(enumerate(CLASS_NAMES)));result['mitre_tactic']=ids.map(dict(enumerate(TACTICS)))
    return result


## 4.4 เตรียมฟังก์ชัน harmonization กลาง

กำหนดขั้นนำกฎกลางไปใช้กับตารางทั้งสาม รวมถึงข้อมูลกำกับว่าแต่ละค่ามาจากอะไร


In [ ]:
def harmonize_tables(dataset, features, labels, metadata, source):
    f=features.reset_index(drop=True).copy()
    l=labels.reset_index(drop=True).copy()
    m=metadata.reset_index(drop=True).copy()
    source=source.reset_index(drop=True)
    if not len(source)==len(f)==len(l)==len(m):raise ValueError('Source/output row count mismatch')
    for col in ['packets_forward','packets_backward']:
        if col not in f:f[col]=np.nan
    protocol_col={'NSL-KDD':'protocol_type','UNSW-NB15':'proto','CIC-IDS2017':'protocol','BoT-IoT':'proto'}[dataset]
    state_col={'NSL-KDD':'flag','UNSW-NB15':'state','CIC-IDS2017':'flow_state','BoT-IoT':'state'}[dataset]
    m['source_protocol']=source[protocol_col].astype('string')
    m['source_connection_state']=source[state_col].astype('string')
    f['protocol']=normalize_protocol(source[protocol_col])
    if dataset in ['NSL-KDD','UNSW-NB15']:
        service=text(source['service'])
        present=service.notna() & ~service.isin(['','-'])
        f['service_group']=service.map(SERVICE_ALIASES).fillna('other')
        m['source_service']=source['service'].astype('string')
        m['service_basis']='source_service_name'
        m['service_identified']=present
        m['has_service']=present
    else:
        port='dest_port' if dataset=='CIC-IDS2017' else 'dport'
        values=source[port] if port in source else pd.Series(pd.NA,index=source.index,dtype='string')
        m['source_service']=values.astype('string')
        if dataset=='BoT-IoT':
            # ARP has no transport port. The mirror represents it as -1.
            missing_arp_port=text(source[protocol_col]).eq('arp') & text(values).isin(['-1','-1.0'])
            values=values.mask(missing_arp_port.fillna(False))
        present=values.notna() & values.astype('string').str.strip().ne('')
        f['service_group']=values.map(port_to_service)
        m['service_basis']=np.where(present,'destination_port_hint','unavailable')
        m['service_identified']=False  # Port 443 is a hint, not proof of TLS.
        m['has_service']=present
    state=text(source[state_col])
    present_state=state.notna() & ~state.isin(['','-'])
    base=state.map(STATE_ALIASES).fillna('other')
    response=f.bytes_backward.gt(0)
    if 'packets_backward' in f:response |= f.packets_backward.gt(0)
    f['conn_state_group']=base.mask(base.eq('no-reply') & response,'other')
    m['state_reclassified']=base.ne(f.conn_state_group)
    m['connection_state_basis']='source_state_and_directional_counters'
    m['has_connection_state']=present_state
    m['has_duration']=f.duration.notna()
    m['has_protocol']=f.protocol.notna()
    m['has_directional_bytes']=f[['bytes_forward','bytes_backward']].notna().all(axis=1)
    m['has_packet_stats']=f[['packets_forward','packets_backward']].notna().all(axis=1)
    for col,out in [('duration','invalid_duration'),('bytes_forward','invalid_bytes_forward'),('bytes_backward','invalid_bytes_backward')]:
        values=pd.to_numeric(f[col],errors='coerce')
        m[out]=values.isna() | ~np.isfinite(values) | values.lt(0)
    flags=quality_flags(f,m)
    for col in QUALITY_FLAGS:m[col]=flags[col]
    m['quality_ok']=~flags.any(axis=1)
    m['quality_issues']=[';'.join(c for c in QUALITY_FLAGS if row[c]) for row in flags.to_dict('records')]
    m['duration_unit']='seconds'
    m=normalize_boolean_metadata(m)
    l['source_attack_category']=l.attack_category.astype('string')
    l=map_labels(l,m)  # One taxonomy for source tables and CENTRAL.
    return f,l,m


_port_to_service = port_to_service


## 4.5 ใช้กฎกลางและตรวจตาราง

เรียก harmonization กับตารางที่รวมแล้ว ตรวจ record_id และดูจำนวนแถวของแต่ละคลาส


In [ ]:
source_frame = pd.concat([pd.read_csv(path, low_memory=False) for path in split_files.values()], ignore_index=True)
features, labels, metadata = harmonize_tables("UNSW-NB15", features, labels, metadata, source_frame)
if not features['record_id'].equals(labels['record_id']) or not features['record_id'].equals(metadata['record_id']):
    raise ValueError('record_id หรือการเรียงแถวของ features/labels/metadata ไม่ตรงกัน')
if features['record_id'].duplicated().any():
    raise ValueError('record_id ซ้ำ; ไม่บันทึก output ที่เชื่อมกลับแบบกำกวม')
print('จำนวนแถว:', len(features))
print('คอลัมน์ features:', len(features.columns))
display(metadata.groupby('source_split').size().rename('rows').to_frame())
display(labels.groupby(['attack_category', 'label_binary']).size().rename('rows').reset_index())


## 5. บันทึกสามตารางและรายงานตรวจสอบ

`features.csv` เก็บพฤติกรรม, `labels.csv` เก็บคำตอบ และ `metadata.csv` เก็บแหล่ง/รหัส/คุณภาพ ทั้งสามเชื่อมกันด้วย `record_id` รายงาน compatibility จะบอกภายหลังว่า feature ใดใช้ร่วมได้จริง ไม่ได้สรุปจากการตั้งชื่อกลางอย่างเดียว

In [5]:
output_dir.mkdir(parents=True, exist_ok=True)
table_rows = []
for table_name, table in [('features', features), ('labels', labels), ('metadata', metadata)]:
    table.to_csv(output_dir / f'{table_name}.csv', index=False, encoding='utf-8')
    table_rows.append({'table': table_name, 'rows': len(table), 'columns': len(table.columns)})

pd.DataFrame(table_rows).to_csv(output_dir / 'harmonization_report.csv', index=False)
pd.DataFrame(quality_rows).to_csv(output_dir / 'harmonization_quality.csv', index=False)

availability_rows = []
for split_name, table in [('train', features_parts[0]), ('test', features_parts[1])]:
    for column in table.columns:
        if column == 'record_id':
            continue
        availability_rows.append({
            'source_dataset': 'UNSW-NB15', 'split': split_name, 'feature': column,
            'rows_available': int(table[column].notna().sum()),
            'rows_total': len(table),
        })
pd.DataFrame(availability_rows).to_csv(output_dir / 'feature_availability.csv', index=False)

# ใช้ manifest ที่ตรวจ checksum แล้วใน cell แรก เป็นหลักฐานของ raw ที่สร้างตารางนี้
saved_manifest.to_csv(output_dir / 'source_manifest.csv', index=False)
provenance = {
    'dataset': 'UNSW-NB15',
    'source_version': registry['datasets']['UNSW-NB15']['version'],
    'registry_version': registry['registry_version'],
    'feature_contract_version': contract['contract_version'],
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'row_count': len(features),
    'source_files': saved_manifest.to_dict(orient='records'),
}
(output_dir / 'provenance.json').write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False), encoding='utf-8'
)

display(pd.DataFrame(table_rows))
print('สร้างไฟล์ใน:', output_dir)
print('รายการไฟล์:', ', '.join(sorted(path.name for path in output_dir.iterdir() if path.suffix.lower() == '.csv')))

,table,rows,columns
0,features,257673,22
1,labels,257673,5
2,metadata,257673,22


สร้างไฟล์ใน: C:\Users\Windows 10\Desktop\CSI_ML_Final\data-pipeline\sub-dataset\UNSW-NB15\output\harmonized
รายการไฟล์: feature_availability.csv, features.csv, harmonization_quality.csv, harmonization_report.csv, labels.csv, metadata.csv, source_manifest.csv


## 6. ตรวจ mapping และดูค่าก่อน-หลังที่สำคัญ

ตาราง mapping แสดงคอลัมน์ UNSW ต้นทาง → ตาราง/คอลัมน์กลาง พร้อมกติกาแปลงและจำนวนแถวที่ได้ค่า แถวที่ไม่พบคอลัมน์หรือค่าถูกปฏิเสธจะไม่ถูกเดาค่าขึ้นมา แต่คงเป็น missing และมี quality flag ส่วน `other` หมายถึงพบค่าแล้วแต่ไม่ได้อยู่ในกลุ่มย่อยที่กำหนด ไม่ได้แปลว่าเป็นข้อมูลเสีย

In [6]:
mapping_specs = [
    ('dur', 'features.duration', 'แปลงเป็นตัวเลขวินาที; ค่าติดลบ/ไม่ใช่ตัวเลข → missing + invalid_duration'),
    ('proto', 'features.protocol', '6→tcp, 17→udp, 1→icmp; โปรโตคอลอื่น → other'),
    ('sbytes', 'features.bytes_forward', 'แปลงตัวเลข; ใช้ทิศทางส่งจากต้นทางตามนิยาม UNSW'),
    ('dbytes', 'features.bytes_backward', 'แปลงตัวเลข; ใช้ทิศทางกลับตามนิยาม UNSW'),
    ('service', 'features.service_group', 'รวม alias เช่น www→http, ssl→https_ssl, domain→dns; ค่าอื่น→other'),
    ('state', 'features.conn_state_group', 'รวม state เป็น established/no-reply/reset/other'),
    ('spkts', 'features.packets_forward', 'คงจำนวน packet ทิศทางต้นทาง'),
    ('dpkts', 'features.packets_backward', 'คงจำนวน packet ทิศทางกลับ'),
    ('sbytes + dbytes + spkts + dpkts', 'features.mean_packet_size', '(bytes สองทิศทางรวม)/(packet สองทิศทางรวม); packet รวมเป็นศูนย์ → missing'),
    ('sttl / dttl', 'features.ttl_forward / ttl_backward', 'คงรายละเอียด TCP/IP; ค่าไม่มี/ติดลบเป็น missing'),
    ('swin / dwin', 'features.tcp_window_forward / tcp_window_backward', 'คงขนาด TCP window; ค่าไม่มี/ติดลบเป็น missing'),
    ('stime', 'metadata.capture_time', 'Unix seconds → เวลา UTC; ไม่ใช้เป็น feature โดยตรง'),
    ('label', 'labels.label_raw + label_binary', 'เก็บ label เดิมและ binary 0/1 แยกจาก features'),
    ('attack_cat', 'labels.label_source_category + attack_category', 'เก็บหมวดเดิมและ map เป็น taxonomy กลาง'),
    ('orig_row_id / _duplicate_flag', 'metadata.orig_row_id / dedup_flag', 'รักษาที่มาและธง duplicate; ไม่ส่งเป็น model feature'),
]
source_columns = set()
for clean_path in split_files.values():
    source_columns.update(pd.read_csv(clean_path, nrows=0).columns)
output_lookup = {}
for table_name, table in [('features', features), ('labels', labels), ('metadata', metadata)]:
    for column in table.columns:
        if column != 'record_id':
            output_lookup[f'{table_name}.{column}'] = int(table[column].notna().sum())
mapping_rows = []
for source_names, target, rule in mapping_specs:
    candidates = [name.strip() for name in source_names.split('/')]
    if source_names == 'sbytes + dbytes + spkts + dpkts':
        actual_source = source_names if {'sbytes', 'dbytes', 'spkts', 'dpkts'}.issubset(source_columns) else 'บางคอลัมน์ไม่มี'
    elif source_names == 'orig_row_id / _duplicate_flag':
        actual_source = ', '.join(name for name in candidates if name in source_columns) or 'orig_row_id/_duplicate_flag ที่ cleansing เพิ่ม'
    else:
        actual_source = ' / '.join(name for name in candidates if name in source_columns) or 'ไม่พบใน header'
    target_details = []
    current_table = None
    for target_name in target.replace('+', '/').split('/'):
        target_name = target_name.strip()
        if '.' in target_name:
            current_table, target_column = target_name.split('.', 1)
        else:
            target_column = target_name
        target_key = f'{current_table}.{target_column}' if current_table else target_column
        available = output_lookup.get(target_key, 0)
        target_details.append(f'{target_key}: {available:,}/{len(features):,} แถว')
    mapping_rows.append({'source_column': actual_source, 'central_column': target,
                         'rule_or_calculation': rule,
                         'output_coverage': '; '.join(target_details)})
display(pd.DataFrame(mapping_rows))

print('ค่าที่ได้หลัง map: protocol, service และสถานะการเชื่อมต่อ')
for split_name, split_features in zip(split_files, features_parts):
    for column in ['protocol', 'service_group', 'conn_state_group']:
        counts = split_features[column].value_counts(dropna=False).rename('rows').reset_index()
        counts.insert(0, 'split', split_name)
        display(counts)
print('แผนที่ label เดิม → taxonomy กลาง (label ไม่ถูกนำเข้า features):')
display(labels.groupby(['label_source_category', 'attack_category', 'label_binary'], dropna=False)
        .size().rename('rows').reset_index().sort_values('rows', ascending=False))
print('แถวที่แปลงตัวเลขไม่ได้หรือค่าติดลบ:')
display(pd.DataFrame(quality_rows))
print('แถวไม่ถูกลบใน harmonization; ตารางทั้งสามยังเชื่อมกันด้วย record_id')

,source_column,central_column,rule_or_calculation,output_coverage
0,dur,features.duration,แปลงเป็นตัวเลขวินาที; ค่าติดลบ/ไม่ใช่ตัวเลข → ...,"features.duration: 257,673/257,673 แถว"
1,proto,features.protocol,"6→tcp, 17→udp, 1→icmp; โปรโตคอลอื่น → other","features.protocol: 257,673/257,673 แถว"
2,sbytes,features.bytes_forward,แปลงตัวเลข; ใช้ทิศทางส่งจากต้นทางตามนิยาม UNSW,"features.bytes_forward: 257,673/257,673 แถว"
3,dbytes,features.bytes_backward,แปลงตัวเลข; ใช้ทิศทางกลับตามนิยาม UNSW,"features.bytes_backward: 257,673/257,673 แถว"
4,service,features.service_group,"รวม alias เช่น www→http, ssl→https_ssl, domain...","features.service_group: 257,673/257,673 แถว"
5,state,features.conn_state_group,รวม state เป็น established/no-reply/reset/other,"features.conn_state_group: 257,673/257,673 แถว"
6,spkts,features.packets_forward,คงจำนวน packet ทิศทางต้นทาง,"features.packets_forward: 257,673/257,673 แถว"
7,dpkts,features.packets_backward,คงจำนวน packet ทิศทางกลับ,"features.packets_backward: 257,673/257,673 แถว"
8,sbytes + dbytes + spkts + dpkts,features.mean_packet_size,(bytes สองทิศทางรวม)/(packet สองทิศทางรวม); pa...,"features.mean_packet_size: 257,673/257,673 แถว"
9,sttl / dttl,features.ttl_forward / ttl_backward,คงรายละเอียด TCP/IP; ค่าไม่มี/ติดลบเป็น missing,"features.ttl_forward: 257,673/257,673 แถว; fea..."


ค่าที่ได้หลัง map: protocol, service และสถานะการเชื่อมต่อ


,split,protocol,rows
0,train,tcp,79946
1,train,udp,63283
2,train,other,32097
3,train,icmp,15


,split,service_group,rows
0,train,other,99479
1,train,dns,47294
2,train,http,18724
3,train,smtp,5058
4,train,ftp,3428
5,train,ssh,1302
6,train,https_ssl,56


,split,conn_state_group,rows
0,train,established,90977
1,train,no-reply,84266
2,train,reset,83
3,train,other,15


,split,protocol,rows
0,test,tcp,43095
1,test,udp,29418
2,test,other,9819


... (3 outputs left out)

,label_source_category,attack_category,label_binary,rows
6,Normal,Normal,0,93000
5,Generic,Generic/Fuzzer,1,58871
3,Exploits,Exploit/U2R,1,44525
4,Fuzzers,Generic/Fuzzer,1,24246
2,DoS,DoS/DDoS,1,16353
7,Reconnaissance,Probe/Recon,1,13987
0,Analysis,Generic/Fuzzer,1,2677
1,Backdoor,Botnet/Backdoor,1,2329
8,Shellcode,Exploit/U2R,1,1511
9,Worms,Botnet/Backdoor,1,174


แถวที่แปลงตัวเลขไม่ได้หรือค่าติดลบ:


,source_dataset,split,rows,duplicate_rows,invalid_duration,invalid_bytes_forward,invalid_bytes_backward
0,UNSW-NB15,train,175341,0,0,0,0
1,UNSW-NB15,test,82332,0,0,0,0


แถวไม่ถูกลบใน harmonization; ตารางทั้งสามยังเชื่อมกันด้วย record_id


## 7. เตรียมรายงานคุณภาพ

กำหนดโครงสร้างรายงานคุณภาพและวิธีเขียนตารางแยกแถวที่ถูกกักไว้


In [ ]:
PROJECT_ROOT = PIPELINE_DIR.parent

NUMERIC = ["duration", "bytes_forward", "bytes_backward"]


CATEGORIES = {
    "protocol": ["tcp", "udp", "icmp", "other"],
    "service_group": ["dns", "ftp", "http", "https_ssl", "smtp", "ssh", "other"],
    "conn_state_group": ["established", "no-reply", "reset", "other"],
}


CORE = NUMERIC + list(CATEGORIES)


def write_quality_outputs(dataset_dir, f,l,m):
    from pathlib import Path
    d=Path(dataset_dir)/'output/harmonized'
    bad=~m.quality_ok
    q=d/'quarantine';q.mkdir(parents=True,exist_ok=True)
    for name,frame in [('features',f),('labels',l),('metadata',m)]:
        frame.loc[bad].to_csv(q/(name+'.csv'),index=False)
    rows=[]
    for split,block in m.groupby('source_split'):
        row={'source_dataset':dataset_dir.name,'split':split,'rows':len(block),
             'eligible_rows':int(block.quality_ok.sum()),'quarantined_rows':int((~block.quality_ok).sum())}
        for col in QUALITY_FLAGS+['invalid_duration','invalid_bytes_forward','invalid_bytes_backward','state_reclassified']:
            row[col]=int(block[col].sum())
        rows.append(row)
    pd.DataFrame(rows).to_csv(d/'harmonization_quality.csv',index=False)
    path=d/'provenance.json'
    import json
    provenance=json.loads(path.read_text(encoding='utf8')) if path.exists() else {}
    provenance.update({'harmonization_schema_version':6,'tables_verified':True,
                       'eligible_rows':int((~bad).sum()),'quarantined_rows':int(bad.sum()),
                       'quality_policy':'Retain all source records; exclude flagged records from CENTRAL and keep separate quarantine tables'})
    write_json(path,provenance)


def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):h.update(block)
    return h.hexdigest()


def write_json(path, value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,ensure_ascii=False,default=lambda x:x.item() if hasattr(x,'item') else str(x)),encoding='utf8')


def raw_manifest(dataset_dir):
    return [{'file':p.relative_to(Path(dataset_dir)/'raw').as_posix(),'bytes':p.stat().st_size,'sha256':digest(p)} for p in sorted((Path(dataset_dir)/'raw').rglob('*')) if p.is_file() and p.suffix.lower() in {'.csv','.gz','.txt'}]


def verify_manifest(dataset_dir, manifest):
    saved=pd.read_csv(manifest).sort_values('file').reset_index(drop=True)
    now=pd.DataFrame(raw_manifest(dataset_dir)).sort_values('file').reset_index(drop=True)
    require(len(now)>0 and saved[['file','sha256']].equals(now[['file','sha256']]),f'Stale raw manifest: {manifest}; rerun cleansing/harmonization')


## 8. เตรียมการตรวจผลลัพธ์และ dependency stamp

กำหนดวิธีอ่านตารางกลับมาตรวจและเก็บ checksum ของไฟล์ที่ใช้


In [ ]:
def read_tables(folder):
    # Source labels can be numeric (UNSW) or text (NSL/CIC). Preserve lexical
    # identifiers consistently across source tables, concatenation and CSV reload.
    types={name:'string' for name in ['record_id','label_raw','label_source_category','source_attack_category','orig_row_id','split_group','source_dataset','source_split','source_protocol','source_connection_state','source_service','service_basis','connection_state_basis','duration_unit','quality_issues','capture_time']}
    f,l,m=tuple(pd.read_csv(Path(folder)/(t+'.csv'),low_memory=False,dtype=types) for t in ['features','labels','metadata'])
    return f,l,normalize_boolean_metadata(m)


def validate_tables(f,l,m,allow_quarantined=False):
    require(len(f)>0 and len(f)==len(l)==len(m),'Empty tables or row-count mismatch')
    for name,frame in [('features',f),('labels',l),('metadata',m)]:
        require('record_id' in frame and frame.record_id.notna().all() and frame.record_id.astype(str).str.strip().ne('').all() and frame.record_id.is_unique,f'{name}: missing/duplicate record_id')
    require(f.record_id.equals(l.record_id) and f.record_id.equals(m.record_id),'Record IDs/order mismatch')
    require(not set(f.columns).intersection({'label','label_raw','label_source_category','attack_category','attack_category_id','label_binary','mitre_tactic','is_attack','source_dataset','sample_weight','data_split','is_synthetic'}),'Target/metadata leakage in features')
    require(set(CORE).issubset(f.columns),'Missing core columns')
    computed=quality_flags(f,m)
    eligible=~computed.any(axis=1)
    if 'quality_ok' in m:
        require(m.quality_ok.notna().all() and m.quality_ok.eq(eligible).all(),'quality_ok disagrees with recomputed quality rules')
        for col in computed:require(col in m and m[col].eq(computed[col]).all(),f'{col}: quality flag mismatch')
    require(allow_quarantined or eligible.all(),'Invalid duration/bytes/protocol or cross-field conflicts: quarantined records are not allowed in model splits')
    checked=f.loc[eligible] if allow_quarantined else f
    for c in NUMERIC:
        z=pd.to_numeric(checked[c],errors='coerce')
        require(np.isfinite(z).all() and z.ge(0).all(),f'{c}: missing, infinite or negative input')
    for c,values in CATEGORIES.items():require(checked[c].notna().all() and checked[c].isin(values).all(),f'{c}: missing/unknown categories')
    response=checked.bytes_backward.gt(0)
    if 'packets_backward' in checked:response |= checked.packets_backward.gt(0)
    require(not (checked.conn_state_group.eq('no-reply') & response).any(),'no-reply contradicts backward traffic')
    for c in ['packets_forward','packets_backward','mean_packet_size','ttl_forward','ttl_backward','tcp_window_forward','tcp_window_backward']:
        if c in checked:
            z=checked[c].dropna()
            require(np.isfinite(z).all() and z.ge(0).all(),f'{c}: invalid measured extension')
    require('label_binary' in l and l.label_binary.isin([0,1]).all(),'Invalid binary labels')
    if 'attack_category_id' in l:
        require(l.attack_category_id.isin(range(6)).all(),'Invalid central classes')
        require(l.label_binary.eq(l.attack_category_id.ne(0).astype(int)).all(),'Binary/multiclass labels disagree')
        require(l.attack_category.eq(l.attack_category_id.map(dict(enumerate(CLASS_NAMES)))).all(),'Class name/ID mismatch')
    if 'sample_weight' in m:require(np.isfinite(m.sample_weight).all() and m.sample_weight.gt(0).all(),'Invalid sample weights')
    for flag in ['invalid_duration','invalid_bytes_forward','invalid_bytes_backward']:
        if flag in m:require(not m.loc[eligible,flag].astype(str).str.lower().isin(['true','1']).any(),f'{flag}: flagged invalid rows')


def code_signature(pipeline_dir):
    p=Path(pipeline_dir)
    files=sorted([*p.glob('*.py'),*p.glob('CENTRAL/*.json'),*p.glob('CENTRAL/*.ipynb'),*p.glob('sub-dataset/*/*.ipynb'),*p.glob('sub-dataset/*/*.py')])
    signature = {f.relative_to(p).as_posix():digest(f) for f in files}
    shared = PROJECT_ROOT / "webapp/Dashboard/backend/src/models/central_contract.py"
    signature["../webapp/Dashboard/backend/src/models/central_contract.py"] = digest(shared)
    return signature


def stamp_dataset(dataset_dir):
    d=Path(dataset_dir);p=d.parent.parent
    f,l,m=read_tables(d/'output/harmonized');validate_tables(f,l,m,allow_quarantined=True)
    verify_manifest(d,d/'output/cleansing/source_manifest.csv')
    verify_manifest(d,d/'output/harmonized/source_manifest.csv')
    cleaned=cleaned_fingerprints(d)
    if 'quality_ok' in m:write_quality_outputs(d,f,l,m)
    origin=d/'raw/source_provenance.json'
    write_json(d/'output/harmonized/dependencies.json',{'raw':raw_manifest(d),'code':code_signature(p),'cleaned':cleaned,'source_provenance':digest(origin) if origin.exists() else None,'tables':{t:digest(d/'output/harmonized'/f'{t}.csv') for t in ['features','labels','metadata']}})


def cleaned_fingerprints(dataset_dir):
    d=Path(dataset_dir)
    names={'NSL-KDD':['KDDTrain_clean.csv','KDDTest_clean.csv','KDDTest-21_clean.csv'],
           'UNSW-NB15':['UNSW_NB15_training-set.csv','UNSW_NB15_testing-set.csv'],
           'CIC-IDS2017':['CIC_IDS2017_training-set.csv','CIC_IDS2017_testing-set.csv'],
           'BoT-IoT':['BoT_IoT_training-set.csv','BoT_IoT_testing-set.csv']}[d.name]
    for name in names:require((d/'output/cleansing'/name).is_file(),f'Missing cleaned file: {d.name}/{name}')
    return {name:digest(d/'output/cleansing'/name) for name in names}


## 9. ตรวจไฟล์และบันทึกผลยืนยัน

ตรวจไฟล์ที่เพิ่งเขียน สร้างรายงานคุณภาพ และบันทึก dependencies.json ให้ CENTRAL ใช้ตรวจที่มา


In [ ]:
stamp_dataset(dataset_dir)
